In [1]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

load_dotenv()

True

In [16]:
URI = "neo4j+ssc://e213fac0.databases.neo4j.io"
AUTH = (os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()

# summary = driver.execute_query("""
#     CREATE (u: User {user_id: $userid})
#     CREATE (t: Track {track_id: $trackid})
#     CREATE (u) - [:APPLIED TO] -> (t)
# """,
#     userid="test002", trackid="track002",
#     database_="e213fac0",
# ).summary

# print("Created {nodes_created} nodes in {time} ms.".format(
#     nodes_created=summary.counters.nodes_created,
#     time=summary.result_available_after
# ))

In [18]:
summary = driver.execute_query("""
    CREATE (u:User {user_id: $userid})
    CREATE (t:Track {track_id: $trackid})
    CREATE (u)-[:ASSIGNED_TO]->(t)
""",
    userid="test002", trackid="track002",
    database_="e213fac0",
).summary

print("Created {nodes_created} nodes in {time} ms.".format(
    nodes_created=summary.counters.nodes_created,
    time=summary.result_available_after
))


Created 2 nodes in 1 ms.


In [25]:
def create_user(driver, user_id, phone_number, age_group, native_language):
    with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
        session.execute_write(
            lambda tx: tx.run("""
                MERGE (u:User {user_id: $user_id})
                SET u.phone_number = $phone_number,
                    u.age_group = $age_group,
                    u.native_language = $native_language,
                    u.created_at = timestamp()
            """,
            user_id=user_id,
            phone_number=phone_number,
            age_group=age_group,
            native_language=native_language
            )
        )

create_user(driver, "test003", "1234567890", "adult", "Hindi")

In [27]:
def seed_tracks(tx):
    tracks = [
        ("articulation_child", "Articulation for children"),
        ("articulation_adult", "Articulation for adults"),
        ("adult_post_therapy", "Post-therapy maintenance"),
        ("hearing_impaired",   "Hearing-impaired adaptation"),
        ("confidence",         "Confidence and fluency"),
    ]
    for track_id, name in tracks:
        tx.run("""
            MERGE (t:Track {track_id: $track_id})
            SET t.name = $name
        """, track_id=track_id, name=name)

with driver.session() as session:
    session.execute_write(seed_tracks)

In [28]:
def assign_tracks(driver, user_id, track_id):
    with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
        session.execute_write(
            lambda tx: tx.run("""
                MATCH (u:User {user_id: $user_id})
                MATCH (t: Track {track_id: $track_id})
                CREATE (u)-[:ASSIGNED_TO]->(t)
            """,
            user_id=user_id,
            track_id = track_id
            )
        )
assign_tracks(driver, "test003", "articulation_adult")

In [35]:
def log_session(driver, user_id, session_id, overall_score, duration_seconds):
    with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
        session.execute_write(
            lambda tx: tx.run("""
                MATCH (u:User {user_id: $user_id})
                CREATE(s: Session{session_id: $session_id,overall_score: $overall_score,duration_seconds: $duration_seconds,timestamp: timestamp()})
                CREATE (u)-[:HAS_SESSION]->(s)
            """,
            user_id = user_id,
            session_id = session_id,
            overall_score = overall_score,
            duration_seconds = duration_seconds
            )
        )
log_session(driver, "test003", "session001", 0.5, 40000)

In [ ]:
def log_phoneme_score(driver, session_id, phoneme_symbol, score_value):
    with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
        session.execute_write(
            lambda tx: tx.run("""
                MATCH (s:Session {session_id: $session_id})
                MERGE (p:Phoneme {symbol: $phoneme_symbol})
                CREATE (sc: Score {value: $score_value, timestamp: timestamp()})
                CREATE (s)-[:CONTAINS_SCORE]->(sc)
                CREATE (sc)-[:FOR_PHONEME]->(p)
            """,
            session_id = session_id,
            phoneme_symbol = phoneme_symbol,
            score_value = float(score_value)
            )
        )

log_phoneme_score(driver, "session001", "/r/", "4")


In [48]:
# def update_weak_phonemes(driver, user_id, phoneme_symbol, thresold=0.6):
#     with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
#         session.execute_write(
#             lambda tx: tx.run("""
#                 MATCH (u:User {user_id: $user_id})
#                 MATCH (p:Phoneme {symbol: $phoneme_symbol})
#                 MATCH (u)-[:HAS_SESSION]->(s)-[:CONTAINS_SCORE]->(sc)-[:FOR_PHONEME]->(p)
#                 WITH u,p, avg(sc.value) as avg_score
#                 WHERE avg_score < $threshold
#                 MERGE (u)-[:WEAK_AT]->(p)
#             """,
#             user_id = user_id,
#             phoneme_symbol = phoneme_symbol,
#             thresold = thresold
#             )
#         )

# update_weak_phonemes(driver, "test003", "/r/")

# def update_weak_phonemes(tx, user_id, phoneme_symbol, threshold=0.6):
#     with driver.session(database=os.getenv("NEO4J_DATABASE")) as session:
#             session.execute_write(
#                 lambda tx: tx.run(""" 
#                     MATCH (u:User {user_id: $user_id})
#                     MATCH (p:Phoneme {symbol: $phoneme_symbol})
#                     MATCH (u)-[:HAS_SESSION]->(s)-[:CONTAINS_SCORE]->(sc)-[:FOR_PHONEME]->(p)
#                     WITH u, p, avg(sc.value) as avg_score
#                     WHERE avg_score < $threshold
#                     MERGE (u)-[:WEAK_AT]->(p)""", user_id=user_id, phoneme_symbol=phoneme_symbol, threshold=threshold))
# update_weak_phonemes(driver, "test003", "/r/")

def update_weak_phonemes(driver, user_id, threshold=0.6):
    with driver.session(database="e213fac0") as session:
        session.execute_write(
            lambda tx: tx.run("""
                MATCH (u:User {user_id: $user_id})
                MATCH (u)-[:HAS_SESSION]->(s)-[:CONTAINS_SCORE]->(sc)-[:FOR_PHONEME]->(p)
                WITH u, p, avg(sc.value) as avg_score
                WHERE avg_score < $threshold
                MERGE (u)-[:WEAK_AT]->(p)
            """,
            user_id=user_id,
            threshold=threshold
            )
        )

update_weak_phonemes(driver, "test003")